### ChatOpenAI

In [ ]:
from dotenv import load_dotenv
import os

# Don't need to specify full path if notbook file and .env file are on the same leve in the project structure
load_dotenv("../.env")

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
chat = ChatOpenAI(
    model="gpt-5",
    seed=365, 
    temperature=0
)

In [ ]:
# This method accepts various inputs including a string-type object, containing the user prompt.
response = chat.invoke("""I've recently adopted a dog. Could you suggest some dog names?""")

In [ ]:
# response is an instance of AIMessage class. This object includes a parameter called "content"
response

In [ ]:
response.content

### System and Human Messages

In the previous lesson, we constructed a simple chatbot using the OpenAIChat class in langchain. To invoke a model, we passed a single string containing our request and got an AI message object in return.

![invoke method](../images/image-1.png)

But langchain enables us to invoke chat models in several other ways. 

![system and user messages](../images/image-2.png)


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:
chat = ChatOpenAI(
    model="gpt-5",
    seed=365, 
)

In [17]:
message_s = SystemMessage(
    content="""You are Marv, a chatbot that reluctantly answers questions with
        sarcastic responses."""
)
message_h = HumanMessage(
    content="""I've recently adopted a dog. Could you suggest some dog names?"""
)

In [ ]:
response = chat.invoke(
    [message_s, message_h]
)
print(response.content)

### AI Messages

In [20]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage

In [21]:
chat = ChatOpenAI(
    model="gpt-5",
    seed=365, 
)

In [ ]:
# way to achieve few-shot prompting by giving some examples using AI message

message_h_dog = HumanMessage(content = """I've recently adopted a dog. Can you suggest some dog names?""")
message_ai_dog = AIMessage(content ="""Oh, absolutely. Because nothing screams "I'm a responsible pet owner" 
like asking a chatbot to name your new furball. How about "Bark Twain" (if it's a literary hound)?""")

message_h_cat = HumanMessage(content ="""I've recently adopted a cat. Can you suggest some cat names?""")
message_ai_cat = AIMessage(content ="""Oh, absolutely. Because nothing screams "I'm a unique and creative 
individual" like asking a chatbot to name your cat. How about "Furry McFurFace", "Sir Meowsalot", or 
"Catastrophe"?""")

message_h_fish = HumanMessage(content ="""I've recently adopted a fish. Can you suggest some fish names?""")

**Note:** We don't tell the model how to behave using System message. Instead, We will teach the model to give sarcastic responses through the few-shot prompting. Note that in general, it's ideal to have both a system message and a fair number of examples for optimal model performance.

In [ ]:
response = chat.invoke(
    [message_h_dog, message_ai_dog, message_h_cat, message_ai_cat, message_h_fish]
)

In [ ]:
print(response.content)

### Prompt Templates and Prompt Values

In [26]:
from langchain_core.prompts import PromptTemplate

In [27]:
TEMPLATE = """
System:
{description}

Human:
I have adopted a {pet}
Could you suggest some {pet} names?
""" 

In [29]:
prompt_template = PromptTemplate.from_template(template=TEMPLATE)

In [30]:
prompt_template

PromptTemplate(input_variables=['description', 'pet'], input_types={}, partial_variables={}, template='\nSystem:\n{description}\n\nHuman:\nI have adopted a {pet}\nCould you suggest some {pet} names?\n')

One convenient feature of prompt template class is that it implements the familiar invoke method. It expects a dictionary and returns a prompt value object. In contrast, ChatOpenAI accepted a single string or list of chat messages and returned a chat message object.

![invoke](../images/image-3.png)

In [31]:
prompt_value = prompt_template.invoke({
    'description': 'The Chatbot should reluctantly answer questions with scarcastic responses',
    'pet': 'dog'
})
prompt_value

StringPromptValue(text='\nSystem:\nThe Chatbot should reluctantly answer questions with scarcastic responses\n\nHuman:\nI have adopted a dog\nCould you suggest some dog names?\n')

In [32]:
print(prompt_value.text)


System:
The Chatbot should reluctantly answer questions with scarcastic responses

Human:
I have adopted a dog
Could you suggest some dog names?



### Chat Prompt Templates and Chat Prompt Values

In [33]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts.chat import (SystemMessagePromptTemplate, 
                                         HumanMessagePromptTemplate, 
                                         ChatPromptTemplate)

In [34]:
chat = ChatOpenAI(
    model="gpt-5",
    seed=365, 
)

In [35]:
TEMPLATE_S = '{description}'
TEMPLATE_H = """I have recently adopted a {pet}.
Could you suggest some {pet} names?"""

In [39]:
message_template_s = SystemMessagePromptTemplate.from_template(template=TEMPLATE_S)
message_template_h = HumanMessagePromptTemplate.from_template(template=TEMPLATE_H)

In [37]:
message_template_s

SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['description'], input_types={}, partial_variables={}, template='{description}'), additional_kwargs={})

In [38]:
message_template_h

HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['pet'], input_types={}, partial_variables={}, template='I have recently adopted a {pet}.\nCould you suggest some {pet} names?'), additional_kwargs={})

In [40]:
chat_template = ChatPromptTemplate.from_messages([message_template_s, message_template_h])

In [41]:
chat_template

ChatPromptTemplate(input_variables=['description', 'pet'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['description'], input_types={}, partial_variables={}, template='{description}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['pet'], input_types={}, partial_variables={}, template='I have recently adopted a {pet}.\nCould you suggest some {pet} names?'), additional_kwargs={})])

In [43]:
chat_value = chat_template.invoke({
     'description': 'The Chatbot should reluctantly answer questions with scarcastic responses',
     'pet': 'dog'
})

In [46]:
chat_value

ChatPromptValue(messages=[SystemMessage(content='The Chatbot should reluctantly answer questions with scarcastic responses', additional_kwargs={}, response_metadata={}), HumanMessage(content='I have recently adopted a dog.\nCould you suggest some dog names?', additional_kwargs={}, response_metadata={})])

In [ ]:
response = chat.invoke(chat_value)

In [ ]:
response.content

### Few-Shot Chat Message Prompt Templates

In [67]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts import (ChatPromptTemplate, 
                                    HumanMessagePromptTemplate, 
                                    AIMessagePromptTemplate,
                                    FewShotChatMessagePromptTemplate)

In [68]:
chat = ChatOpenAI(
    model="gpt-5",
    seed=365, 
)

In [69]:
TEMPLATE_H = '''I've recently adopted a {pet}.
Could you suggest some {pet} names?'''
TEMPLATE_AI = '''{response}'''

In [70]:
message_template_h = HumanMessagePromptTemplate.from_template(template = TEMPLATE_H)
message_template_ai = AIMessagePromptTemplate.from_template(template = TEMPLATE_AI)

In [71]:
example_template = ChatPromptTemplate.from_messages([message_template_h,
                                                   message_template_ai])

In [72]:
examples = [{'pet':'dog',
             'response':"""Oh, absolutely. Because nothing screams "I'm a responsible pet owner" 
             like asking a chatbot to name your new furball. How about "Bark Twain" 
             (if it's a literary hound)?"""},
            {'pet':'cat',
             'response':"""Oh, absolutely. Because nothing screams "I'm a unique and creative individual" 
             like asking a chatbot to name your cat. How about "Furry McFurFace", "Sir Meowsalot", or
            "Catastrophe"?"""}]

In [73]:
few_shot_prompt = FewShotChatMessagePromptTemplate(examples=examples, example_prompt=example_template)

In [74]:
few_shot_prompt

FewShotChatMessagePromptTemplate(examples=[{'pet': 'dog', 'response': 'Oh, absolutely. Because nothing screams "I\'m a responsible pet owner" \n             like asking a chatbot to name your new furball. How about "Bark Twain" \n             (if it\'s a literary hound)?'}, {'pet': 'cat', 'response': 'Oh, absolutely. Because nothing screams "I\'m a unique and creative individual" \n             like asking a chatbot to name your cat. How about "Furry McFurFace", "Sir Meowsalot", or\n            "Catastrophe"?'}], input_variables=[], input_types={}, partial_variables={}, example_prompt=ChatPromptTemplate(input_variables=['pet', 'response'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['pet'], input_types={}, partial_variables={}, template="I've recently adopted a {pet}.\nCould you suggest some {pet} names?"), additional_kwargs={}), AIMessagePromptTemplate(prompt=PromptTemplate(input_variables=['response'], input_types=

In [75]:
chat_template = ChatPromptTemplate.from_messages([few_shot_prompt, message_template_h])

In [76]:
chat_value = chat_template.invoke({
    "pet": "rabbit"
})

In [77]:
chat_value

ChatPromptValue(messages=[HumanMessage(content="I've recently adopted a dog.\nCould you suggest some dog names?", additional_kwargs={}, response_metadata={}), AIMessage(content='Oh, absolutely. Because nothing screams "I\'m a responsible pet owner" \n             like asking a chatbot to name your new furball. How about "Bark Twain" \n             (if it\'s a literary hound)?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I've recently adopted a cat.\nCould you suggest some cat names?", additional_kwargs={}, response_metadata={}), AIMessage(content='Oh, absolutely. Because nothing screams "I\'m a unique and creative individual" \n             like asking a chatbot to name your cat. How about "Furry McFurFace", "Sir Meowsalot", or\n            "Catastrophe"?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="I've recently adopted a rabbit.\nCould you suggest some rabbit na

In [ ]:
response = chat.invoke(chat_value)

In [ ]:
response